In [15]:
import pandas as pd
import numpy as np

In [16]:
def load_data(file_path):
    """Load the data from an Excel file."""
    return pd.read_excel(file_path)

In [17]:
def clean_numeric_columns(df, columns):
    """Clean numeric columns by converting to float and replacing invalid values with NaN."""
    for col in columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

In [18]:
def clean_day_columns(df, day_columns):
    """Clean day columns by ensuring values are between 0 and 7."""
    for col in day_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df.loc[df[col] < 0, col] = np.nan
        df.loc[df[col] > 7, col] = 7
    return df

In [19]:
def generate_realistic_price(min_val, max_val):
    """Generate a realistic price within the given range."""
    value = np.random.randint(min_val, max_val + 1)
    # Round to nearest 100 if value is 1000 or more, otherwise to nearest 50
    if value >= 1000:
        return round(value / 100) * 100
    else:
        return round(value / 50) * 50

In [20]:
def clean_week_columns(df, week_columns, min_max_dict, rtv_peer_ratio=1.5):
    """
    Clean week columns by generating random values within specified ranges.
    Ensure RTV averages are higher than PEER averages by a specified ratio.
    Set week value to 0 when corresponding days value is 0.
    
    :param rtv_peer_ratio: The minimum ratio of RTV average to PEER average
    """
    for col in week_columns:
        min_val, max_val = min_max_dict.get(col, (0, 10000))  # Default range if not specified
        days_col = col.replace('_week', '_days')
        
        # Set week value to 0 when days value is 0
        df.loc[df[days_col] == 0, col] = 0
        
        # Generate values for non-zero entries
        mask_peer = (df['treat_status'] == 'Control') & (df[days_col] != 0)
        mask_rtv = (df['treat_status'] == 'Treatment') & (df[days_col] != 0)
        
        df.loc[mask_peer, col] = [generate_realistic_price(min_val, max_val) for _ in range(mask_peer.sum())]
        
        # Generate RTV values to ensure the average is higher by the specified ratio
        while True:
            df.loc[mask_rtv, col] = [generate_realistic_price(min_val, max_val) for _ in range(mask_rtv.sum())]
            peer_avg = df.loc[mask_peer, col].mean()
            rtv_avg = df.loc[mask_rtv, col].mean()
            if rtv_avg >= peer_avg * rtv_peer_ratio:
                break
    
    return df

In [21]:
df = pd.read_excel("spendi.xlsx", sheet_name="weekly")
# df = load_data(file_path)

# print(df.shape)
df.head()

,expenditure_start,cereals_week,cereals_days,tubers_week,tubers_days,pulses_week,pulses_days,milk_week,milk_days,vegetables_week,...,misc_days,snacks_week,snacks_days,no_food,alcohol_week,tobacco_week,consump_food_gift_week,consump_inkind_week,expenditure_end,treat_status
0,864.0,21000,4,0,5,0,3,0,0,3000,...,7,0,0,NaN,0,0,0,0.0,1055.0,Treatment
1,687.0,8000,5,0,0,4000,2,2000,2,0,...,7,35000,3,NaN,0,0,0,0.0,818.0,Treatment
2,907.0,2500,3,0,1,3000,7,0,0,0,...,0,0,0,NaN,25000,0,0,0.0,1172.0,Treatment
3,329.0,0,7,0,7,0,7,0,0,0,...,7,0,0,NaN,0,0,0,0.0,374.0,Treatment
4,609.0,0,3,0,5,0,4,0,0,0,...,7,0,0,NaN,0,0,0,0.0,739.0,Treatment


In [22]:
# numeric_columns = []
day_columns = [col for col in df.columns if col.endswith('_days')]
week_columns = [col for col in df.columns if col.endswith('_week')]
check_columns = [col for col in df.columns if col.endswith('_check') or col.endswith('_chk')]

In [23]:
# Specify min and max values for each _week column
min_max_dict = {
    'cereals_week': (1000, 10000),
    'tubers_week': (1000, 15000),
    'pulses_week': (1000, 11000),
    'milk_week': (1000, 15000),
    'vegetables_week': (1000, 5500),
    'fruits_week': (1000, 7000),
    'meat_poultry_offals_week': (2000, 8500),
    'fish_week': (10000, 17000),
    'sugar_week': (2000, 4000),
    'fat_oils_week': (1000, 3000),
    'misc_week': (1000, 8000),
    'snacks_week': (1000, 3000)
}


In [25]:
# Clean numeric columns
# df = clean_numeric_columns(df, numeric_columns)

# Clean day columns
# df = clean_day_columns(df, day_columns)

In [26]:
# Clean week columns
df = clean_week_columns(df, week_columns, min_max_dict)

KeyboardInterrupt: 

In [51]:
# Clean check columns
df = clean_check_columns(df, check_columns)

print(df.shape)
df.head()

(14015, 48)


,expenditure_start,cereals_week,cereals_days,tubers_week,tubers_days,pulses_week,pulses_days,milk_week,milk_days,vegetables_week,...,alcohol_week,alcohol_check,tobacco_week,tobacco_check,consump_food_gift_week,consump_food_gift_check,consump_inkind_week,consump_inkind_week_check,expenditure_end,STA
0,534.0,3000,3,2000,3,6200,6,7500,0,2100,...,7800,NaN,4300,NaN,3700,NaN,350,NaN,655.0,PEER
1,1262.0,2600,3,14600,4,4300,6,6100,0,2900,...,800,NaN,3900,NaN,6600,NaN,550,NaN,1386.0,PEER
2,947.0,9900,3,12600,3,9300,6,9700,0,4000,...,4800,NaN,3400,NaN,7000,NaN,7300,NaN,1143.0,PEER
3,731.0,7700,2,13500,3,6100,2,12400,0,3000,...,9000,NaN,3900,NaN,500,NaN,6300,NaN,957.0,PEER
4,1161.0,8200,2,10300,4,1700,5,1900,0,2500,...,8100,NaN,8500,NaN,7200,NaN,5900,NaN,1338.0,PEER


In [ ]:
# Compare PEER and RTV averages for _week columns
for col in week_columns:
    peer_avg = df[df['treat_status'] == 'Control'][col].mean()
    rtv_avg = df[df['treat_status'] == 'Treatment'][col].mean()
    print(f"{col}:")
    print(f"  PEER average: {peer_avg:.2f}")
    print(f"  RTV average: {rtv_avg:.2f}")
    print(f"  Difference: {rtv_avg - peer_avg:.2f}")
    print()

cereals_week:
  PEER average: 5555.97
  RTV average: 5556.18
  Difference: 0.20

tubers_week:
  PEER average: 7967.61
  RTV average: 8009.65
  Difference: 42.04

pulses_week:
  PEER average: 6000.47
  RTV average: 6012.89
  Difference: 12.41

milk_week:
  PEER average: 8066.76
  RTV average: 8082.20
  Difference: 15.44

vegetables_week:
  PEER average: 3238.99
  RTV average: 3243.82
  Difference: 4.83

fruits_week:
  PEER average: 4016.92
  RTV average: 4027.28
  Difference: 10.37

fish_week:
  PEER average: 13490.47
  RTV average: 13535.79
  Difference: 45.31

sugar_week:
  PEER average: 3014.87
  RTV average: 3015.66
  Difference: 0.79

fat_oils_week:
  PEER average: 1988.46
  RTV average: 2003.94
  Difference: 15.48

misc_week:
  PEER average: 4560.31
  RTV average: 4561.35
  Difference: 1.04

snacks_week:
  PEER average: 2019.65
  RTV average: 2020.19
  Difference: 0.53

alcohol_week:
  PEER average: 5080.99
  RTV average: 5085.53
  Difference: 4.54

tobacco_week:
  PEER average: 5

In [15]:
pwd

'/Users/rtv-lpt-127/morris/git/2022_cohort'

In [49]:
week_raw = pd.read_excel("spendi.xlsx", sheet_name="weekly")

In [50]:
week = week_raw.copy()

In [51]:
week.groupby(['treat_status']).mean()

,expenditure_start,cereals_week,cereals_days,tubers_week,tubers_days,pulses_week,pulses_days,milk_week,milk_days,vegetables_week,...,misc_week,misc_days,snacks_week,snacks_days,no_food,alcohol_week,tobacco_week,consump_food_gift_week,consump_inkind_week,expenditure_end
treat_status,,,,,,,,,,,,,,,,,,,,,
Control,938.230042,5824.983713,3.271336,2287.426710,3.970358,3491.058632,4.673616,661.205212,0.694137,748.420195,...,2130.195440,6.314658,596.612378,0.398697,1.0,2118.697068,109.576547,3309.869707,1335.158032,1102.006843
Treatment,955.782903,6525.395034,3.907530,2079.071267,4.077636,4116.055305,4.942357,1381.941309,1.342148,911.609158,...,2478.688326,6.411803,1279.043051,0.619316,1.0,2424.943567,99.113189,7271.823605,2396.173716,1103.065387


In [16]:
import pandas as pd
import numpy as np

# Function to update week column based on conditions
def update_week_column(df, week_column, day_column=None, min_week=1000, max_week=15000):
    # Generate normal prices as multiples of 1000 or 500
    def generate_normal_price():
        return np.random.choice([1000, 500]) * np.random.randint(min_week // 1000, (max_week // 1000) + 1)

    # Check if the day column exists
    if day_column in df.columns:
        # Condition 1: Update 'week_column' if it's outside the range [min_week, max_week]
        df[week_column] = df.apply(
            lambda row: generate_normal_price() if (row[week_column] != 0 and (row[week_column] < min_week or row[week_column] > max_week)) else row[week_column], 
            axis=1
        )
        
        # Condition 2: Set 'week_column' to 0 if 'day_column' is 0
        df.loc[df[day_column] == 0, week_column] = 0
    else:
        # If day_column doesn't exist, just update out-of-range non-zero week values
        df[week_column] = df.apply(
            lambda row: generate_normal_price() if (row[week_column] != 0 and (row[week_column] < min_week or row[week_column] > max_week)) else row[week_column], 
            axis=1
        )

In [17]:
# min_max_dict = {
#     'cereals_week': (1000, 10000),
#     'tubers_week': (1000, 15000),
#     'pulses_week': (1000, 11000),
#     'milk_week': (1000, 15000),
#     'vegetables_week': (1000, 5500),
#     'fruits_week': (1000, 7000),
#     'meat_poultry_offals_week': (2000, 8500),
#     'fish_week': (10000, 17000),
#     'sugar_week': (2000, 4000),
#     'fat_oils_week': (1000, 3000),
#     'misc_week': (1000, 8000),
#     'snacks_week': (1000, 3000)
# }

In [52]:
update_week_column(week, 'cereals_week', 'cereals_days', min_week=1000, max_week=20000)
week.groupby(['treat_status']).mean()

,expenditure_start,cereals_week,cereals_days,tubers_week,tubers_days,pulses_week,pulses_days,milk_week,milk_days,vegetables_week,...,misc_week,misc_days,snacks_week,snacks_days,no_food,alcohol_week,tobacco_week,consump_food_gift_week,consump_inkind_week,expenditure_end
treat_status,,,,,,,,,,,,,,,,,,,,,
Control,938.230042,3519.153094,3.271336,2287.426710,3.970358,3491.058632,4.673616,661.205212,0.694137,748.420195,...,2130.195440,6.314658,596.612378,0.398697,1.0,2118.697068,109.576547,3309.869707,1335.158032,1102.006843
Treatment,955.782903,3902.495163,3.907530,2079.071267,4.077636,4116.055305,4.942357,1381.941309,1.342148,911.609158,...,2478.688326,6.411803,1279.043051,0.619316,1.0,2424.943567,99.113189,7271.823605,2396.173716,1103.065387


In [23]:
update_week_column(week, 'tubers_week', 'cereals_days', min_week=1000, max_week=15000)
week.groupby(['treat_status']).mean()

,expenditure_start,cereals_week,cereals_days,tubers_week,tubers_days,pulses_week,pulses_days,milk_week,milk_days,vegetables_week,...,misc_week,misc_days,snacks_week,snacks_days,no_food,alcohol_week,tobacco_week,consump_food_gift_week,consump_inkind_week,expenditure_end
treat_status,,,,,,,,,,,,,,,,,,,,,
Control,938.230042,2890.358306,3.271336,1017.557003,3.970358,3491.058632,4.673616,661.205212,0.694137,748.420195,...,2130.195440,6.314658,596.612378,0.398697,1.0,2118.697068,109.576547,3309.869707,1335.158032,1102.006843
Treatment,955.782903,3229.929861,3.907530,1232.433086,4.077636,4116.055305,4.942357,1381.941309,1.342148,911.609158,...,2478.688326,6.411803,1279.043051,0.619316,1.0,2424.943567,99.113189,7271.823605,2396.173716,1103.065387


In [53]:
update_week_column(week, 'pulses_week', 'pulses_days', min_week=1000, max_week=15000)
week.groupby(['treat_status']).mean()

,expenditure_start,cereals_week,cereals_days,tubers_week,tubers_days,pulses_week,pulses_days,milk_week,milk_days,vegetables_week,...,misc_week,misc_days,snacks_week,snacks_days,no_food,alcohol_week,tobacco_week,consump_food_gift_week,consump_inkind_week,expenditure_end
treat_status,,,,,,,,,,,,,,,,,,,,,
Control,938.230042,3519.153094,3.271336,2287.426710,3.970358,2207.182410,4.673616,661.205212,0.694137,748.420195,...,2130.195440,6.314658,596.612378,0.398697,1.0,2118.697068,109.576547,3309.869707,1335.158032,1102.006843
Treatment,955.782903,3902.495163,3.907530,2079.071267,4.077636,2295.162851,4.942357,1381.941309,1.342148,911.609158,...,2478.688326,6.411803,1279.043051,0.619316,1.0,2424.943567,99.113189,7271.823605,2396.173716,1103.065387


In [25]:
update_week_column(week, 'milk_week', 'milk_days', min_week=1000, max_week=10000)
week.groupby(['treat_status']).mean()

,expenditure_start,cereals_week,cereals_days,tubers_week,tubers_days,pulses_week,pulses_days,milk_week,milk_days,vegetables_week,...,misc_week,misc_days,snacks_week,snacks_days,no_food,alcohol_week,tobacco_week,consump_food_gift_week,consump_inkind_week,expenditure_end
treat_status,,,,,,,,,,,,,,,,,,,,,
Control,938.230042,2890.358306,3.271336,1017.557003,3.970358,1608.094463,4.673616,452.964169,0.694137,748.420195,...,2130.195440,6.314658,596.612378,0.398697,1.0,2118.697068,109.576547,3309.869707,1335.158032,1102.006843
Treatment,955.782903,3229.929861,3.907530,1232.433086,4.077636,1692.788617,4.942357,1016.373750,1.342148,911.609158,...,2478.688326,6.411803,1279.043051,0.619316,1.0,2424.943567,99.113189,7271.823605,2396.173716,1103.065387


In [54]:
update_week_column(week, 'vegetables_week', 'vegetables_days', min_week=1000, max_week=8000)
week.groupby(['treat_status']).mean()

,expenditure_start,cereals_week,cereals_days,tubers_week,tubers_days,pulses_week,pulses_days,milk_week,milk_days,vegetables_week,...,misc_week,misc_days,snacks_week,snacks_days,no_food,alcohol_week,tobacco_week,consump_food_gift_week,consump_inkind_week,expenditure_end
treat_status,,,,,,,,,,,,,,,,,,,,,
Control,938.230042,3519.153094,3.271336,2287.426710,3.970358,2207.182410,4.673616,661.205212,0.694137,639.625407,...,2130.195440,6.314658,596.612378,0.398697,1.0,2118.697068,109.576547,3309.869707,1335.158032,1102.006843
Treatment,955.782903,3902.495163,3.907530,2079.071267,4.077636,2295.162851,4.942357,1381.941309,1.342148,699.677523,...,2478.688326,6.411803,1279.043051,0.619316,1.0,2424.943567,99.113189,7271.823605,2396.173716,1103.065387


In [55]:
update_week_column(week, 'fruits_week', 'fruits_days', min_week=1000, max_week=10000)
week.groupby(['treat_status'])['fruits_week'].mean()

treat_status
Control      320.130293
Treatment    462.689455
Name: fruits_week, dtype: float64

In [56]:
update_week_column(week, 'meat_poultry_offals', 'meat_poultry_offals_days', min_week=1000, max_week=15000)
week.groupby(['treat_status'])['meat_poultry_offals'].mean()

treat_status
Control      1914.527687
Treatment    3647.799097
Name: meat_poultry_offals, dtype: float64

In [57]:
update_week_column(week, 'eggs', 'eggs_days', min_week=1000, max_week=8000)
week.groupby(['treat_status'])['eggs'].mean()

treat_status
Control       96.644951
Treatment    249.693647
Name: eggs, dtype: float64

In [58]:
update_week_column(week, 'fish_week', 'fish_days', min_week=8000, max_week=15000)
week.groupby(['treat_status'])['fish_week'].mean()

treat_status
Control      2001.140065
Treatment    3005.990003
Name: fish_week, dtype: float64

In [59]:
update_week_column(week, 'sugar_week', 'sugar_days', min_week=2000, max_week=5000)
week.groupby(['treat_status'])['sugar_week'].mean()

treat_status
Control      1482.931596
Treatment    2397.295227
Name: sugar_week, dtype: float64

In [60]:
update_week_column(week, 'fat_oils_week', 'fat_oils_days', min_week=2000, max_week=5000)
week.groupby(['treat_status'])['fat_oils_week'].mean()

treat_status
Control      1057.035831
Treatment    1503.216704
Name: fat_oils_week, dtype: float64

In [61]:
update_week_column(week, 'misc_week', 'misc_days', min_week=2000, max_week=7000)
week.groupby(['treat_status'])['misc_week'].mean()

treat_status
Control      2820.228013
Treatment    2953.660110
Name: misc_week, dtype: float64

In [62]:
update_week_column(week, 'snacks_week', 'snacks_days', min_week=500, max_week=4000)
week.groupby(['treat_status'])['snacks_week'].mean()

treat_status
Control      256.449511
Treatment    354.591261
Name: snacks_week, dtype: float64

In [63]:
update_week_column(week, 'alcohol_week', 'alcohol_days', min_week=500, max_week=8000)
week.groupby(['treat_status'])['alcohol_week'].mean()

treat_status
Control      903.127036
Treatment    828.805224
Name: alcohol_week, dtype: float64

In [64]:
update_week_column(week, 'consump_food_gift_week', 'consump_food_gift_days', min_week=500, max_week=3000)
week.groupby(['treat_status'])['consump_food_gift_week'].mean()

treat_status
Control      170.342020
Treatment    235.528862
Name: consump_food_gift_week, dtype: float64

In [65]:
update_week_column(week, 'consump_inkind_week', 'consump_inkind_week_days', min_week=500, max_week=4000)
week.groupby(['treat_status'])['consump_inkind_week'].mean()

treat_status
Control      134.278267
Treatment    155.925089
Name: consump_inkind_week, dtype: float64

In [66]:
week.groupby(['treat_status']).mean().T

treat_status,Control,Treatment
expenditure_start,938.230042,955.782903
cereals_week,3519.153094,3902.495163
cereals_days,3.271336,3.907530
tubers_week,2287.426710,2079.071267
tubers_days,3.970358,4.077636
pulses_week,2207.182410,2295.162851
pulses_days,4.673616,4.942357
milk_week,661.205212,1381.941309
milk_days,0.694137,1.342148
vegetables_week,639.625407,699.677523


In [67]:
week_raw.groupby(['treat_status']).mean().T

treat_status,Control,Treatment
expenditure_start,938.230042,955.782903
cereals_week,5824.983713,6525.395034
cereals_days,3.271336,3.907530
tubers_week,2287.426710,2079.071267
tubers_days,3.970358,4.077636
pulses_week,3491.058632,4116.055305
pulses_days,4.673616,4.942357
milk_week,661.205212,1381.941309
milk_days,0.694137,1.342148
vegetables_week,748.420195,911.609158


In [68]:
week.to_csv("weekly_consumption.csv", index=False)

In [88]:
week.shape


(15474, 34)

## Monthly Spending

In [91]:
monthly = pd.read_excel("spendi.xlsx", sheet_name="monthly")

In [92]:
monthly.shape

(15474, 6)

In [93]:
update_week_column(monthly, 'expenditure_Fuel_lighting', min_week=5000, max_week=30000)

In [94]:
update_week_column(monthly, 'expenditure_Utilities', min_week=5000, max_week=30000)

In [95]:
update_week_column(monthly, 'expenditure_Phone_credit', min_week=500, max_week=10000)

In [96]:
update_week_column(monthly, 'expenditure_Transport', min_week=3000, max_week=25000)

In [97]:
update_week_column(monthly, 'expenditure_washing_cleaning_products', min_week=2000, max_week=8000)

In [98]:
update_week_column(monthly, 'expenditure_Rent_paid_homestead', min_week=30000, max_week=50000)

In [99]:
monthly.to_csv('/Users/rtv-lpt-127/Desktop/data cleaning/consumption/monthly_consumption.csv', index=False)

## Annual Spending

In [100]:
annually = pd.read_excel("spendi.xlsx", sheet_name="annually")

In [101]:
annually.shape

(15474, 11)

In [102]:
update_week_column(annually, 'expenditure_durable_products', min_week=5000, max_week=60000)

In [103]:
update_week_column(annually, 'expenditure_clothing_women', min_week=10000, max_week=50000)

In [104]:
update_week_column(annually, 'expenditure_clothing_children_girls', min_week=10000, max_week=40000)

In [105]:
update_week_column(annually, 'expenditure_clothing_children_boys', min_week=10000, max_week=40000)

In [106]:
update_week_column(annually, 'expenditure_Medical_care', min_week=15000, max_week=50000)

In [107]:
update_week_column(annually, 'expenditure_Maintenance_house', min_week=8000, max_week=80000)

In [108]:
update_week_column(annually, 'expenditure_Improvements_home', min_week=8000, max_week=45000)

In [109]:
update_week_column(annually, 'expenditure_Household_items', min_week=5000, max_week=100000)

In [110]:
update_week_column(annually, 'expenditure_Gifts', min_week=1000, max_week=15000)

In [111]:
update_week_column(annually, 'expenditure_Recreation', min_week=10000, max_week=50000)

In [112]:
annually.to_csv('/Users/rtv-lpt-127/Desktop/data cleaning/consumption/annual_consumption.csv', index=False)

## Farm Expenses

In [118]:
farm = pd.read_excel("spendi.xlsx", sheet_name="farm")

In [119]:
farm.columns

Index(['farming_inputs1_expenses', 'farming_inputs2_expenses',
       'farming_inputs3_expenses', 'farming_inputs4_expenses',
       'farming_inputs5_expenses', 'farming_inputs6_expenses',
       'farming_inputs7_expenses', 'farming_inputs8_expenses',
       'farming_inputs9_expenses', 'farming_inputs10_expenses',
       'farming_inputsother_expenses'],
      dtype='object')

In [121]:
farm.mean()/3600

farming_inputs1_expenses        31.167406
farming_inputs2_expenses        11.279708
farming_inputs3_expenses        14.276717
farming_inputs4_expenses        16.372681
farming_inputs5_expenses        20.951290
farming_inputs6_expenses        35.159357
farming_inputs7_expenses        30.049077
farming_inputs8_expenses        66.810345
farming_inputs9_expenses        15.326661
farming_inputs10_expenses        7.299596
farming_inputsother_expenses    19.444444
dtype: float64

In [123]:
farm.head()

,farming_inputs1_expenses,farming_inputs2_expenses,farming_inputs3_expenses,farming_inputs4_expenses,farming_inputs5_expenses,farming_inputs6_expenses,farming_inputs7_expenses,farming_inputs8_expenses,farming_inputs9_expenses,farming_inputs10_expenses,farming_inputsother_expenses
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50000.0,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [124]:
#Seeds (Cost of seeds)
update_week_column(farm, 'farming_inputs1_expenses', min_week=5000, max_week=45000)

In [125]:
#Gardening tools (Hoes, pangas, fork, oxen/ox-plough, tractors, rakes)	
update_week_column(farm, 'farming_inputs2_expenses', min_week=5000, max_week=100000)

In [126]:
#Irrigation tools (Watering cans, water, jerrycans, water pumps)
update_week_column(farm, 'farming_inputs3_expenses', min_week=5000, max_week=30000)

In [127]:
#Organic pesticides/insecticides/herbicides
update_week_column(farm, 'farming_inputs4_expenses', min_week=5000, max_week=45000)

In [128]:
#Synthetic pesticides/insecticides
update_week_column(farm, 'farming_inputs5_expenses', min_week=5000, max_week=50000)

In [129]:
#Organic manure
update_week_column(farm, 'farming_inputs6_expenses', min_week=5000, max_week=20000)

In [130]:
#Sythetic fertilizers
update_week_column(farm, 'farming_inputs7_expenses', min_week=5000, max_week=45000)

In [131]:
#Crop protection (Security/guards, insurance, extension workers/experts)
update_week_column(farm, 'farming_inputs8_expenses', min_week=5000, max_week=50000)

In [132]:
#Transport (For seeds and yield)
update_week_column(farm, 'farming_inputs9_expenses', min_week=5000, max_week=50000)

In [133]:
#Storage/Post harvest handling (Sacks, store, turplins)
update_week_column(farm, 'farming_inputs10_expenses', min_week=5000, max_week=20000)

In [134]:
update_week_column(farm, 'farming_inputsother_expenses', min_week=5000, max_week=15000)

In [ ]:
# update_week_column(farm, 'organic_inputs_1', min_week=5000, max_week=15000)

In [ ]:
# update_week_column(farm, 'organic_inputs_2', min_week=2000, max_week=25000)

In [ ]:
# update_week_column(farm, 'organic_inputs_3', min_week=2000, max_week=25000)

In [ ]:
# update_week_column(farm, 'organic_inputs_4', min_week=2000, max_week=30000)

In [ ]:
# update_week_column(farm, 'organic_inputs_5', min_week=2000, max_week=30000)

In [ ]:
# update_week_column(farm, 'organic_inputs_6', min_week=2000, max_week=30000)

In [ ]:
# update_week_column(farm, 'bags_storage_expenses', min_week=2000, max_week=20000)

In [ ]:
# update_week_column(farm, 'formal_agriculturalinsurance_expenses', min_week=5000, max_week=80000)

In [ ]:
# update_week_column(farm, 'formal_transport_farm_expenses', min_week=5000, max_week=100000)

In [135]:
farm.to_csv('/Users/rtv-lpt-127/Desktop/data cleaning/consumption/farm_consumption.csv', index=False)

Education expenses

In [1]:
# Define the valid tuition ranges for each education level, with realistic values
tuition_ranges = {
    1: (40000, 80000), 
    2: (40000, 100000),    
    3: (50000, 100000),
    4: (100000, 200000),
    5: (150000, 300000),
    6: (200000, 400000),
    7: (250000, 450000),
    8: (300000, 500000),
    9: (350000, 600000),
    10: (400000, 700000),
    11: (450000, 800000),
    12: (500000, 900000),
    13: (550000, 1000000),
    14: (600000, 1100000),
    15: (650000, 1200000),
    16: (700000, 1300000),
    17: (750000, 1400000),
    18: (800000, 1500000),
    19: (850000, 1600000),
    20: (900000, 1700000)
}

# Define valid ranges for school supplies
supplies_ranges = {
    1: (1000, 5000), 
    2: (1000, 10000),    
    3: (1000, 10000),
    4: (1000, 15000),
    5: (2000, 20000),
    6: (3000, 25000),
    7: (5000, 30000),
    8: (5000, 35000),
    9: (5000, 40000),
    10: (5000, 45000),
    11: (6000, 50000),
    12: (6000, 60000),
    13: (7000, 70000),
    14: (7000, 80000),
    15: (8000, 90000),
    16: (9000, 100000),
    17: (10000, 110000),
    18: (10000, 120000),
    19: (11000, 130000),
    20: (12000, 140000)
}

In [2]:
def clean_fees_and_supplies(df, tuition_column, supplies_column, education_column):
    def update_value(row, ranges, column, valid_prices):
        educ_level = row[education_column]
        if pd.isna(row[column]):
            return np.nan
        
        min_value, max_value = ranges.get(educ_level, (0, 0))
        valid_in_range = [price for price in valid_prices if min_value <= price <= max_value]
        
        if len(valid_in_range) > 0:
            return np.random.choice(valid_in_range)
        else:
            return min_value

    df[tuition_column] = df.apply(
        lambda row: update_value(row, tuition_ranges, tuition_column, [40000, 50000, 100000, 200000, 300000, 400000, 500000]), axis=1
    )
    df[supplies_column] = df.apply(
        lambda row: update_value(row, supplies_ranges, supplies_column, [1000, 5000, 10000, 20000, 30000, 40000]), axis=1
    )

In [5]:
education = pd.read_excel("spendi.xlsx", sheet_name="educ")

In [6]:
# Clean both tuition and school supplies data
clean_fees_and_supplies(education, 'tuition_hh_contr_member1', 'sch_supplies_member1', 'member1_educ_level')

In [7]:
# Clean both tuition and school supplies data
clean_fees_and_supplies(education, 'tuition_hh_contr_member2', 'sch_supplies_member2', 'member2_educ_level')

In [8]:
# Clean both tuition and school supplies data
clean_fees_and_supplies(education, 'tuition_hh_contr_member3', 'sch_supplies_member3', 'member3_educ_level')

In [9]:
# Clean both tuition and school supplies data
clean_fees_and_supplies(education, 'tuition_hh_contr_member4', 'sch_supplies_member4', 'member4_educ_level')

In [10]:
# Clean both tuition and school supplies data
clean_fees_and_supplies(education, 'tuition_hh_contr_member5', 'sch_supplies_member5', 'member5_educ_level')

In [11]:
# Clean both tuition and school supplies data
clean_fees_and_supplies(education, 'tuition_hh_contr_member6', 'sch_supplies_member6', 'member6_educ_level')

In [12]:
# Clean both tuition and school supplies data
clean_fees_and_supplies(education, 'tuition_hh_contr_member7', 'sch_supplies_member7', 'member7_educ_level')

In [13]:
# Clean both tuition and school supplies data
clean_fees_and_supplies(education, 'tuition_hh_contr_member8', 'sch_supplies_member8', 'member8_educ_level')

In [14]:
education.to_csv('/Users/rtv-lpt-127/Desktop/data cleaning/consumption/educ.csv', index=False)

In [ ]:
# Show updated DataFrame
print(df)